# The Agent Loop (ReAct)

**WatSPEED Agentic AI prep — Week 1-2 - agent fundamentals**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## From one tool call to an agent

Notebook 02 did a single call. An **agent** is that step in a `while` loop: the model
keeps requesting tools until it decides it can answer. The classic framing is
**ReAct** — *Reason, then Act* — where the model interleaves thinking with tool use.

The loop is about fifteen lines. Frameworks add retries, streaming and tracing on top,
but if you can write this you can debug any of them.

### The "Why": Why we need dynamic loops (ReAct)

> **The Legacy Friction:** Traditional data pipelines (SAS Macros, standard Python scripts) are rigid, linear tracks. If step 2 fails because a file is missing or a parameter is slightly off, the entire script crashes and requires human intervention to fix.
>
> **The RAP Value Proposition:** The ReAct (Reason + Act) loop gives your code autonomy. Instead of crashing when data is missing, the agent *reasons* about the error, *acts* by using a tool to go find the missing data, and continues the pipeline. It transforms a fragile, linear script into a robust, self-healing system.


In [2]:
tools = ToolBox()

@tools.tool("Count respondents matching a filter", schema(column='string', value='string'))
def count_where(column: str, value: str) -> dict:
    table = {("age_group", "18-29"): 412, ("age_group", "60+"): 388}
    return {"n": table.get((column, value), 0)}

@tools.tool("Mean trust score (1-5) for a subgroup", schema(column='string', value='string'))
def mean_trust(column: str, value: str) -> dict:
    table = {("age_group", "18-29"): 2.81, ("age_group", "60+"): 3.44}
    return {"mean_trust": table.get((column, value))}

print(tools.describe())

- count_where(column, value): Count respondents matching a filter
- mean_trust(column, value): Mean trust score (1-5) for a subgroup


### The loop itself

Three things worth noticing:

- **`max_steps` is not optional.** Without it a confused model loops forever and bills you.
- Every tool result is appended to `messages`. The growing list *is* the agent's memory
  within a run — which is why context management (notebook 04) matters so fast.
- The loop exits when the model returns `content` instead of `tool_calls`.

In [3]:
def run_agent(llm, tools: ToolBox, question: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content": f"You answer survey questions using tools:\n{tools.describe()}"},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        resp = llm.chat(messages, tools=tools.schemas())

        if not resp.wants_tool:
            trace(step, "model", f"final answer: {resp.content}")
            return resp.content

        for call in resp.tool_calls:
            trace(step, "model", f"call {call['name']}({call['arguments']})")
            try:
                result = tools.call(call["name"], call["arguments"])
            except Exception as exc:                    # feed errors back, don't crash
                result = {"error": f"{type(exc).__name__}: {exc}"}
            trace(step, "tool", f"-> {result}")
            messages.append({"role": "assistant", "tool_calls": [call]})
            messages.append({"role": "tool", "name": call["name"], "content": str(result)})

    return "[stopped: hit max_steps without a final answer]"

In [4]:
llm = get_llm([
    LLMResponse(tool_calls=[{"name": "mean_trust", "arguments": {"column": "age_group", "value": "18-29"}}]),
    LLMResponse(tool_calls=[{"name": "mean_trust", "arguments": {"column": "age_group", "value": "60+"}}]),
    LLMResponse(content="Under-30s average 2.81 vs 3.44 for 60+, so trust rises with age."),
])

banner("Agent transcript")
answer = run_agent(llm, tools, "Do younger and older respondents differ in AI trust?")
print("\nRETURNED:", answer)

Using stub-llm (deterministic, offline) - no OPENAI_API_KEY found, so results are scripted.

Agent transcript
  [ 1] model        | call mean_trust({'column': 'age_group', 'value': '18-29'})
  [ 1] tool         | -> {'mean_trust': 2.81}
  [ 2] model        | call mean_trust({'column': 'age_group', 'value': '60+'})
  [ 2] tool         | -> {'mean_trust': 3.44}
  [ 3] model        | final answer: Under-30s average 2.81 vs 3.44 for 60+, so trust rises with age.

RETURNED: Under-30s average 2.81 vs 3.44 for 60+, so trust rises with age.


### Failure modes you will actually hit

Two runs below. The first shows a tool erroring — the agent should recover, not crash.
The second shows a model that never stops asking. Both are the reason `max_steps`
and the `try/except` exist.

In [5]:
banner("1. Tool raises - error is fed back as an observation")
llm_err = get_llm([
    LLMResponse(tool_calls=[{"name": "no_such_tool", "arguments": {}}]),
    LLMResponse(content="That tool does not exist; I used the available ones instead."),
], verbose=False)
run_agent(llm_err, tools, "trigger an error")

banner("2. Model never converges - max_steps saves you")
llm_loop = get_llm([LLMResponse(
    tool_calls=[{"name": "count_where", "arguments": {"column": "age_group", "value": "18-29"}}]
)] * 20, verbose=False)
print("\nRETURNED:", run_agent(llm_loop, tools, "loop forever", max_steps=3))


1. Tool raises - error is fed back as an observation
  [ 1] model        | call no_such_tool({})
  [ 1] tool         | -> {'error': 'KeyError: "No tool named \'no_such_tool\'. Available: [\'count_where\', \'mean_trust\']"'}
  [ 2] model        | final answer: That tool does not exist; I used the available ones instead.

2. Model never converges - max_steps saves you
  [ 1] model        | call count_where({'column': 'age_group', 'value': '18-29'})
  [ 1] tool         | -> {'n': 412}
  [ 2] model        | call count_where({'column': 'age_group', 'value': '18-29'})
  [ 2] tool         | -> {'n': 412}
  [ 3] model        | call count_where({'column': 'age_group', 'value': '18-29'})
  [ 3] tool         | -> {'n': 412}

RETURNED: [stopped: hit max_steps without a final answer]


### The same loop in LangChain

```python
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(ChatOpenAI(model="gpt-4o-mini"), [crosstab, mean_trust])
agent.invoke({"messages": [("user", "Do younger and older respondents differ?")]})
```

One line replaces `run_agent`. It runs the loop you just wrote, plus retries and
tracing. You now know what to look for when its output surprises you.

---
### Try it yourself

1. Add a step counter to the system prompt so the model knows its budget.
2. Change the loop to run *all* `tool_calls` in one step in parallel. When is that wrong?
3. What breaks if you forget to append the `assistant` message before the `tool` message?